In [2]:
from pathlib import Path
import shutil
import re

source_root = Path("/Volumes/TOSHIBA_EXT/02-Raw_data-anat/longitudinal_freesurfer_149")
destination_root = Path("/Volumes/TOSHIBA_EXT/03-Raw_fMRI/2025")
destination_root.mkdir(parents=True, exist_ok=True)

def strip_prefix(name):
    return re.sub(r"^\d+[-_]?", "", name.lower())

def is_t1_folder(name):
    clean = strip_prefix(name)
    return "3d_t1" in clean or ("3d" in clean and "t1" in clean)

def is_memorytask_folder(name):
    clean = strip_prefix(name)
    return "memorytask" in clean

def is_fieldmap_folder(name):
    clean = strip_prefix(name)
    return "fieldmap" in clean or ("gre" in clean and "mapping" in clean)

def should_copy(name):
    return is_t1_folder(name) or is_memorytask_folder(name) or is_fieldmap_folder(name)

visit_dirs = sorted(
    folder for folder in source_root.iterdir()
    if folder.is_dir() and re.match(r"^\d+_irm_t\d+$", folder.name.lower())
)

copied_folders = []
empty_matches = []

for visit_dir in visit_dirs:
    selected_subfolders = [
        sub for sub in sorted(visit_dir.iterdir())
        if sub.is_dir() and should_copy(sub.name)
    ]

    if not selected_subfolders:
        empty_matches.append(visit_dir.name)
        continue

    destination_visit_dir = destination_root / visit_dir.name
    destination_visit_dir.mkdir(parents=True, exist_ok=True)

    for subfolder in selected_subfolders:
        destination_subfolder = destination_visit_dir / subfolder.name
        shutil.copytree(subfolder, destination_subfolder, dirs_exist_ok=True)
        copied_folders.append(str(destination_subfolder))

print(f"Processed participant visits: {len(visit_dirs)}")
print(f"Copied selected subfolders: {len(copied_folders)}")
if empty_matches:
    print("No matching subfolders found for:")
    for visit_name in empty_matches:
        print(visit_name)

KeyboardInterrupt: 